<a href="https://colab.research.google.com/github/andersonmoraix/analisedeconjuntura_anotacoes/blob/main/aula05_analisePNANcontinua.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CURSO: Análise de Conjuntura usando Python - Análise Macro<br>
AULA: Coleta e tratamento de dados da PNAD Contínua com Python<br>
INFORMAÇÕES: saiba mais sobre a Pesquisa Nacional por Amostra de Domicílios Contínua (PNADC/IBGE) no link https://www.ibge.gov.br/estatisticas/sociais/trabalho/2511-np-pnad-continua/30980-pnadc-divulgacao-pnadc4.html <br>
AUTOR: Fernando da Silva/Cientista de dados

# Bibliotecas

In [ ]:
# Instalar bibliotecas
!pip install sidrapy

# Importar bibliotecas
import sidrapy as sidra
import pandas as pd

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


# Dados

In [ ]:
# Coleta de dados
dados_brutos = sidra.get_table(
    table_code = "6318",
    territorial_level = "1",
    ibge_territorial_code = "all",
    variable = "1641",
    classifications = {"629": "32385,32386,32387,32446,32447"},
    period = "all"
)
dados_brutos

,NC,NN,MC,MN,V,D1C,D1N,D2C,D2N,D3C,D3N,D4C,D4N
0,Nível Territorial (Código),Nível Territorial,Unidade de Medida (Código),Unidade de Medida,Valor,Brasil (Código),Brasil,Trimestre Móvel (Código),Trimestre Móvel,Variável (Código),Variável,Condição em relação à força de trabalho e cond...,Condição em relação à força de trabalho e cond...
1,1,Brasil,1572,Mil pessoas,153601,1,Brasil,201203,jan-fev-mar 2012,1641,Pessoas de 14 anos ou mais de idade,32385,Total
2,1,Brasil,1572,Mil pessoas,95664,1,Brasil,201203,jan-fev-mar 2012,1641,Pessoas de 14 anos ou mais de idade,32386,Força de trabalho
3,1,Brasil,1572,Mil pessoas,88011,1,Brasil,201203,jan-fev-mar 2012,1641,Pessoas de 14 anos ou mais de idade,32387,Força de trabalho - ocupada
4,1,Brasil,1572,Mil pessoas,7653,1,Brasil,201203,jan-fev-mar 2012,1641,Pessoas de 14 anos ou mais de idade,32446,Força de trabalho - desocupada
...,...,...,...,...,...,...,...,...,...,...,...,...,...
626,1,Brasil,1572,Mil pessoas,173328,1,Brasil,202208,jun-jul-ago 2022,1641,Pessoas de 14 anos ou mais de idade,32385,Total
627,1,Brasil,1572,Mil pessoas,108706,1,Brasil,202208,jun-jul-ago 2022,1641,Pessoas de 14 anos ou mais de idade,32386,Força de trabalho
628,1,Brasil,1572,Mil pessoas,99013,1,Brasil,202208,jun-jul-ago 2022,1641,Pessoas de 14 anos ou mais de idade,32387,Força de trabalho - ocupada
629,1,Brasil,1572,Mil pessoas,9694,1,Brasil,202208,jun-jul-ago 2022,1641,Pessoas de 14 anos ou mais de idade,32446,Força de trabalho - desocupada


In [ ]:
# Tratamento de dados
(
    dados_brutos.rename(columns = dados_brutos.iloc[0])
    .rename(
        columns = {
            "Trimestre Móvel (Código)": "data",
            "Condição em relação à força de trabalho e condição de ocupação": "condicao",
            "Valor": "valor"
            }
        )
    .query("valor not in 'Valor'")
    .assign(
        data = lambda x: pd.to_datetime(x.data, format = "%Y%m"),
        valor = lambda x: x.valor.astype(float)
        )
    .pivot(index = "data", columns = "condicao", values = "valor") #para alterar o formato da tabela de long para wide
    .assign(
        tx_desocupacao = lambda x: x["Força de trabalho - desocupada"] / x["Força de trabalho"] * 100 #validar as colunas se estao como numericas antes de executar a funcao, alterar para float
        )
)

condicao,Fora da força de trabalho,Força de trabalho,Força de trabalho - desocupada,Força de trabalho - ocupada,Total,tx_desocupacao
data,,,,,,
2012-03-01,57937.0,95664.0,7653.0,88011.0,153601.0,7.999875
2012-04-01,57411.0,96380.0,7534.0,88846.0,153791.0,7.816974
2012-05-01,57164.0,96823.0,7444.0,89379.0,153987.0,7.688256
2012-06-01,57169.0,97010.0,7363.0,89647.0,154180.0,7.589939
2012-07-01,57299.0,97076.0,7290.0,89786.0,154375.0,7.509580
...,...,...,...,...,...,...
2022-04-01,64946.0,107861.0,11349.0,96512.0,172807.0,10.521875
2022-05-01,64791.0,108147.0,10631.0,97516.0,172938.0,9.830139
2022-06-01,64719.0,108349.0,10080.0,98269.0,173068.0,9.303270
